# Prior Sensitivity & Expert Scaling (GWT)

We've established that the DCM is **prior-dominated** at the current data size (4 experts): the support/demandingness-derived Beta priors (all with concentration $\alpha + \beta = 10$) barely shift from prior to posterior. This means the hand-specified evidential labels, not the empirical ratings, are the primary driver of $P(\text{consciousness} = 1)$.

**This notebook asks:** would collecting more expert data fix this? Specifically:
1. How does $P(\text{consciousness} = 1 \mid \text{data})$ change as we scale from 1 to 6 experts?
2. At what point (if any) do the Beta posteriors begin to depart meaningfully from their priors?
3. Is the trajectory suggesting that a realistic increase in expert count would overcome prior domination, or would we need an order-of-magnitude more data?

For $k > 4$ we bootstrap synthetic experts from the existing pool, giving a **conservative lower bound** on the effect of genuinely new experts.

In [ ]:
import sys, os, copy, warnings

sys.path.insert(0, "..")
os.chdir(os.path.join(os.path.dirname(os.path.abspath("."))))
warnings.filterwarnings("ignore", message="Loop fusion failed")

import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
from collections import defaultdict

from dcm_model import (
    ModelConfig,
    OrdinalDataProcessor,
    EvidenceProcessor,
    BayesianModelBuilder,
    load_data,
    node_key,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

---
## 1. Prior inspection

Every node's conditional probability is drawn from a $\text{Beta}(\alpha, \beta)$ with $\alpha + \beta = 10$. The table below shows representative prior shapes. A concentration of 10 is equivalent in information to roughly 10 pseudo-observations.

In [ ]:
config = ModelConfig()
ep = EvidenceProcessor(config)

combos = [
    ("strong support", "neutral"),
    ("moderate support", "neutral"),
    ("strong support", "moderately demanding"),
    ("strong support", "weakly demanding"),
    ("no bearing", "neutral"),
]

fig, axes = plt.subplots(1, len(combos), figsize=(3 * len(combos), 2.5), sharey=True)
x = np.linspace(0.001, 0.999, 200)

for ax, (s, d) in zip(axes, combos):
    ap, bp, aa, ba = ep.get_beta_parameters(s, d)
    ax.plot(
        x,
        sp_stats.beta.pdf(x, ap, bp),
        label=f"present ({ap:.1f},{bp:.1f})",
        color="steelblue",
    )
    ax.plot(
        x,
        sp_stats.beta.pdf(x, aa, ba),
        label=f"absent ({aa:.1f},{ba:.1f})",
        color="salmon",
    )
    ax.set_title(f"{s}\n{d}", fontsize=8)
    ax.set_xlabel("p")
    ax.legend(fontsize=6)

axes[0].set_ylabel("Density")
fig.suptitle("Support/demandingness Beta priors (concentration = 10)", fontsize=11)
fig.tight_layout()
plt.show()

---
## 2. Data setup & expert subsampling

GWT has 4 real experts. To project beyond the current data, we bootstrap synthetic experts (k=5, 6) by resampling ratings from the existing pool with replacement. This gives a **conservative lower bound** -- real new experts would bring genuinely independent information.

In [ ]:
# Full data
all_data = load_data(config)
stance_data = next(s for s in all_data if s["name"] == config.TARGET_STANCE)

full_processor = OrdinalDataProcessor(config)
full_processor.process(stance_data, config.TARGET_SYSTEM)

n_real = len(full_processor.expert_names)
print(f"Real experts ({n_real}): {full_processor.expert_names}")
print(f"Anchor: {full_processor.anchor_expert}")
print(f"Indicators with observations: {len(full_processor.observations)}")
print(f"Total ratings: {sum(len(v) for v in full_processor.observations.values())}")

In [ ]:
def make_processor_with_k_experts(
    full_proc: OrdinalDataProcessor, k: int, seed: int = 42
) -> OrdinalDataProcessor:
    """Create a processor with observations from the first k experts.

    For k > n_real, bootstrap additional synthetic experts by resampling
    ratings from the existing pool.
    """
    rng = np.random.default_rng(seed)
    proc = OrdinalDataProcessor(full_proc.config)

    n_real = len(full_proc.expert_names)
    n_synth = max(0, k - n_real)
    k_real = min(k, n_real)

    # Expert names: real[:k_real] + synthetic
    proc.expert_names = list(full_proc.expert_names[:k_real])
    for s in range(n_synth):
        proc.expert_names.append(f"Synthetic_{s + 1}")
    proc.expert_to_idx = {name: i for i, name in enumerate(proc.expert_names)}
    proc.anchor_expert = proc.expert_names[0]

    # Filter observations to first k_real experts
    for nkey, obs_list in full_proc.observations.items():
        filtered = [(eidx, rating) for eidx, rating in obs_list if eidx < k_real]
        if not filtered and n_synth == 0:
            continue
        proc.observations[nkey] = list(filtered)

    # Add synthetic experts by resampling from existing ratings
    if n_synth > 0:
        for nkey, obs_list in full_proc.observations.items():
            all_ratings = [r for _, r in obs_list]
            if not all_ratings:
                continue
            for s in range(n_synth):
                synth_idx = k_real + s
                sampled_rating = int(rng.choice(all_ratings))
                proc.observations[nkey].append((synth_idx, sampled_rating))

    # Remove empty indicators
    proc.observations = defaultdict(
        list, {k: v for k, v in proc.observations.items() if v}
    )
    return proc


# Quick check
for k in range(1, 7):
    p = make_processor_with_k_experts(full_processor, k)
    n_obs = sum(len(v) for v in p.observations.values())
    tag = "" if k <= n_real else f" ({k - n_real} synthetic)"
    print(f"k={k}: {len(p.expert_names)} experts{tag}, {n_obs} total ratings")

---
## 3. Fit models across expert counts

We fit the ordinal DCM for $k = 1, 2, \ldots, 6$ experts. Using reduced sampling (500/500, 2 chains) to keep total runtime manageable.

In [ ]:
import time

k_values = [1, 2, 3, 4, 5, 6]
results = {}  # k -> {idata, builder, processor}

for k in k_values:
    print(f"\n{'=' * 40} k={k} {'=' * 40}")
    t0 = time.time()

    fit_config = ModelConfig(
        NUM_SAMPLES=500,
        NUM_TUNE=500,
        NUM_CHAINS=2,
        TARGET_ACCEPT=0.95,
    )
    proc = make_processor_with_k_experts(full_processor, k)
    evidence_proc = EvidenceProcessor(fit_config)
    builder = BayesianModelBuilder(fit_config, evidence_proc, proc)
    model = builder.build_model(stance_data)
    idata = builder.sample(model)

    results[k] = {"idata": idata, "builder": builder, "processor": proc}
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.0f}s")

---
## 4. P(consciousness = 1) vs number of experts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

means, lows, highs = [], [], []
for k in k_values:
    idata = results[k]["idata"]
    stance_varname = results[k]["builder"].node_to_varname[config.TARGET_STANCE]
    draws = idata.posterior[f"{stance_varname}_bern"].values.flatten()
    means.append(float(draws.mean()))
    lows.append(float(np.percentile(draws, 3)))
    highs.append(float(np.percentile(draws, 97)))

ax.plot(k_values, means, "o-", color="steelblue", label="Posterior mean")
ax.fill_between(k_values, lows, highs, alpha=0.2, color="steelblue", label="94% CI")
ax.axvline(
    n_real, color="grey", linestyle="--", alpha=0.5, label=f"Real experts (n={n_real})"
)

# Prior reference
prior_mean = config.DEFAULT_ALPHA / (config.DEFAULT_ALPHA + config.DEFAULT_BETA)
ax.axhline(
    prior_mean,
    color="salmon",
    linestyle=":",
    alpha=0.7,
    label=f"Prior mean ({prior_mean:.2f})",
)

ax.set_xlabel("Number of experts")
ax.set_ylabel("P(consciousness = 1 | data)")
ax.set_title("Stance posterior vs expert count (GWT)")
ax.set_xticks(k_values)
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

for k in k_values:
    tag = "*" if k > n_real else " "
    print(
        f"  k={k}{tag}: P(C=1) = {means[k_values.index(k)]:.3f}  [{lows[k_values.index(k)]:.3f}, {highs[k_values.index(k)]:.3f}]"
    )
print("  * = includes bootstrapped synthetic experts")

---
## 5. Prior-posterior comparison for Beta parameters

For a few representative features, we compare the prior $\text{Beta}(\alpha, \beta)$ density with the posterior draws. If posteriors closely track priors, the data is not moving the model much.

In [ ]:
# Pick 4 representative features (first 4 top-level features in GWT)
feature_nodes = [
    e for e in stance_data["evidencers"] if e["type"].lower() == "feature"
][:4]

fig, axes = plt.subplots(
    len(feature_nodes),
    len(k_values),
    figsize=(2.5 * len(k_values), 2.5 * len(feature_nodes)),
    sharex=True,
    sharey="row",
)
x = np.linspace(0.001, 0.999, 200)

for row, feat in enumerate(feature_nodes):
    support = feat.get("support", "no bearing")
    demand = feat.get("demandingness", "neutral")
    ap, bp, aa, ba = ep.get_beta_parameters(support, demand)

    for col, k in enumerate(k_values):
        ax = axes[row, col]
        builder_k = results[k]["builder"]
        idata_k = results[k]["idata"]

        # Find the varname for this feature
        feat_key = None
        for nkey in builder_k.node_to_varname:
            if nkey.endswith(feat["name"]):
                feat_key = nkey
                break
        if feat_key is None:
            continue
        varname = builder_k.node_to_varname[feat_key]

        # Prior
        ax.plot(x, sp_stats.beta.pdf(x, ap, bp), "k--", alpha=0.4, linewidth=1)

        # Posterior for beta_pres
        pres_name = f"{varname}_beta_pres"
        if pres_name in idata_k.posterior:
            post_draws = idata_k.posterior[pres_name].values.flatten()
            ax.hist(
                post_draws,
                bins=30,
                density=True,
                alpha=0.6,
                color="steelblue",
                edgecolor="white",
            )

        if row == 0:
            ax.set_title(f"k={k}", fontsize=9)
        if col == 0:
            ax.set_ylabel(feat["name"][:25], fontsize=8)
        if row == len(feature_nodes) - 1:
            ax.set_xlabel("p")

fig.suptitle(
    "Beta_present posterior (blue) vs prior (dashed) across expert counts",
    fontsize=11,
    y=1.01,
)
fig.tight_layout()
plt.show()

---
## 6. Posterior shift from prior (all features)

Quantify prior-domination: for each feature's `beta_pres` and `beta_abs`, compute the shift between posterior mean and prior mean. Small shifts = prior-dominated.

In [ ]:
# Compute shifts for all features at k=4 (all real experts)
idata_full = results[n_real]["idata"]
builder_full = results[n_real]["builder"]

shifts = []
for feat in feature_nodes:
    support = feat.get("support", "no bearing")
    demand = feat.get("demandingness", "neutral")
    ap, bp, aa, ba = ep.get_beta_parameters(support, demand)

    feat_key = None
    for nkey in builder_full.node_to_varname:
        if nkey.endswith(feat["name"]):
            feat_key = nkey
            break
    if feat_key is None:
        continue
    varname = builder_full.node_to_varname[feat_key]

    for suffix, a, b, label in [
        ("_beta_pres", ap, bp, "present"),
        ("_beta_abs", aa, ba, "absent"),
    ]:
        vname = f"{varname}{suffix}"
        if vname in idata_full.posterior:
            post_mean = float(idata_full.posterior[vname].mean())
            prior_mean = a / (a + b)
            shifts.append(
                {
                    "feature": feat["name"][:30],
                    "param": label,
                    "prior_mean": prior_mean,
                    "post_mean": post_mean,
                    "shift": post_mean - prior_mean,
                }
            )

print(f"{'Feature':<32} {'Param':<8} {'Prior':>6} {'Post':>6} {'Shift':>7}")
print("-" * 65)
for s in shifts:
    print(
        f"{s['feature']:<32} {s['param']:<8} {s['prior_mean']:>6.3f} {s['post_mean']:>6.3f} {s['shift']:>+7.3f}"
    )

all_shifts = [abs(s["shift"]) for s in shifts]
print(f"\nMean |shift|: {np.mean(all_shifts):.4f}")
print(f"Max  |shift|: {np.max(all_shifts):.4f}")

---
## 7. Indicator posteriors across expert counts

Track how selected indicator $P(z_j = 1 \mid \text{data})$ values evolve as we add experts.

In [ ]:
# Pick indicators: highest, lowest, and two mid-range from k=4 results
builder_ref = results[n_real]["builder"]
idata_ref = results[n_real]["idata"]

all_indicators = []
for nkey, varname in builder_ref.node_to_varname.items():
    pz1_name = f"{varname}_pz1"
    if pz1_name in idata_ref.posterior:
        mean_val = float(idata_ref.posterior[pz1_name].mean())
        all_indicators.append((nkey, mean_val))

all_indicators.sort(key=lambda x: x[1])
n_ind = len(all_indicators)
selected = [
    all_indicators[0],  # lowest
    all_indicators[n_ind // 3],  # lower-mid
    all_indicators[2 * n_ind // 3],  # upper-mid
    all_indicators[-1],  # highest
]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for ax, (nkey, _) in zip(axes.flat, selected):
    ind_means, ind_lows, ind_highs = [], [], []
    for k in k_values:
        bk = results[k]["builder"]
        ik = results[k]["idata"]
        vname = bk.node_to_varname.get(nkey)
        if vname and f"{vname}_pz1" in ik.posterior:
            draws = ik.posterior[f"{vname}_pz1"].values.flatten()
            ind_means.append(float(draws.mean()))
            ind_lows.append(float(np.percentile(draws, 3)))
            ind_highs.append(float(np.percentile(draws, 97)))
        else:
            ind_means.append(np.nan)
            ind_lows.append(np.nan)
            ind_highs.append(np.nan)

    ax.plot(k_values, ind_means, "o-", color="steelblue")
    ax.fill_between(k_values, ind_lows, ind_highs, alpha=0.2, color="steelblue")
    ax.axvline(n_real, color="grey", linestyle="--", alpha=0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(k_values)
    short_name = nkey.split(" > ")[-1][:35]
    ax.set_title(short_name, fontsize=9)
    ax.set_ylabel("P(present | data)")

for ax in axes[1]:
    ax.set_xlabel("Number of experts")

fig.suptitle("Selected indicator posteriors vs expert count", fontsize=11)
fig.tight_layout()
plt.show()

---
## 8. Discussion

**What to look for in the results above:**

1. **Rate of posterior departure.** The Beta priors have concentration 10 (~10 pseudo-observations). Each real expert contributes roughly 1 observation per indicator. So at $k=4$, data is ~4 vs ~10 pseudo-observations -- the prior should still dominate. At $k=6$ (even bootstrapped), is the ratio shifting enough to matter?

2. **P(consciousness=1) trajectory.** If still moving between $k=4$ and $k=6$, more data would meaningfully change conclusions. If flat, the hierarchy is already saturated by priors and even substantially more experts won't help without changing prior concentrations.

3. **Asymmetry across features.** Features with more indicators feeding them may update faster than those with few. This could inform where to prioritise data collection.

**Implications for the project:**

- If the trajectory suggests ~20-50 experts are needed to escape prior domination, this motivates either (a) a serious data collection effort, or (b) revisiting the prior concentration parameter (currently hard-coded to 10) as a modelling choice that could be calibrated or treated as uncertain.
- If even bootstrapped experts shift posteriors noticeably, real independent experts would shift them more -- making data collection a high-value intervention.
- The support/demandingness labels remain the most consequential modelling decision in the prior-dominated regime. Any sensitivity analysis should focus there before structural model changes.